In [1]:
import pandas as pd
import pymysql
from typing import Union, List

def fetch_unique_indicators(
    db_info: dict,
    table_name: str = "Korea_company_valuation_ver2"
) -> list:
    """
    DB 테이블에서 indicator 컬럼의 unique 값 리스트 반환
    """

    conn = pymysql.connect(
        host=db_info["host"],
        port=db_info["port"],
        user=db_info["user"],
        password=db_info["password"],
        database=db_info["database"],
        charset="utf8mb4"
    )

    try:
        sql = f"""
        SELECT DISTINCT indicator
        FROM {table_name}
        ORDER BY indicator
        """
        df = pd.read_sql(sql, conn)
    finally:
        conn.close()

    return df["indicator"].tolist()

def fetch_indicator_pivot(
    db_info: dict,
    indicator: Union[str, List[str]],
    forecast_date: str,
    ticker: str,
    table_name: str = "Korea_company_valuation_ver2"
) -> pd.DataFrame:
    """
    Parameters
    ----------
    db_info : dict
    indicator : str or list[str]
        예: "psr_ETS" 또는 ["psr_ETS", "psr_SARIMA"]
    forecast_date : str
        예: "2025-12-05"
    ticker : str
        예: "A005930"
    table_name : str

    Returns
    -------
    DataFrame
        index   : date
        columns : indicator
        values  : value
    """

    # indicator를 리스트로 통일
    if isinstance(indicator, str):
        indicator = [indicator]

    conn = pymysql.connect(
        host=db_info["host"],
        port=db_info["port"],
        user=db_info["user"],
        password=db_info["password"],
        database=db_info["database"],
        charset="utf8mb4"
    )

    try:
        indicator_placeholders = ",".join(["%s"] * len(indicator))

        sql = f"""
        SELECT
            date,
            indicator,
            value
        FROM {table_name}
        WHERE forecast_date = %s
          AND ticker = %s
          AND indicator IN ({indicator_placeholders})
        ORDER BY date
        """

        params = [forecast_date, ticker] + indicator
        df = pd.read_sql(sql, conn, params=params)

    finally:
        conn.close()

    if df.empty:
        return pd.DataFrame()

    pivot_df = (
        df.pivot_table(
            index="date",
            columns="indicator",
            values="value",
            aggfunc="last"
        )
        .sort_index()
    )

    return pivot_df



def get_valuation_pivot_by_forecast_date(
    db_info: dict,
    forecast_date: str,
    indicator: str,
    table_name: str = "Korea_company_valuation_ver2"
) -> pd.DataFrame:
    """
    forecast_date + indicator를 기준으로
    index=date, columns=ticker, values=value 형태의 pivot DataFrame 생성
    """

    conn = pymysql.connect(
        host=db_info["host"],
        port=db_info.get("port", 3306),
        user=db_info["user"],
        password=db_info["password"],
        database=db_info["database"],
        charset="utf8mb4"
    )

    try:
        sql = f"""
        SELECT
            date,
            ticker,
            value
        FROM {table_name}
        WHERE forecast_date = %s
          AND indicator = %s
        ORDER BY date, ticker
        """
        df = pd.read_sql(sql, conn, params=[forecast_date, indicator])
    finally:
        conn.close()

    if df.empty:
        raise ValueError(
            f"No data found for forecast_date={forecast_date}, indicator={indicator}"
        )

    pivot_df = (
        df.pivot(index="date", columns="ticker", values="value")
          .sort_index()
    )

    return pivot_df


def _to_numeric_df(df: pd.DataFrame) -> pd.DataFrame:
    """DF 값 전체를 숫자로 강제 변환 (쉼표 포함 문자열 처리)."""
    out = df.copy()
    out = out.applymap(lambda x: str(x).replace(",", "").strip() if isinstance(x, str) else x)
    out = out.apply(pd.to_numeric, errors="coerce")
    return out

def _keep_quarter_end_rows(df: pd.DataFrame) -> pd.DataFrame:
    """
    분기말(3/6/9/12월) + 월말인 날짜만 남김.
    (예: 2025-03-31, 2025-06-30, 2025-09-30, 2025-12-31)
    """
    idx = df.index
    is_q_month = idx.month.isin([3, 6, 9, 12])
    is_month_end = idx.is_month_end
    return df.loc[is_q_month & is_month_end].copy()

def load_indicator_pivot(
    db_info: dict,
    indicator: str,
    forecast_date: str,
    table_name: str = "Korea_company_valuation_ver2",
    agg: str = "last"  # "first", "last", "mean", "sum" 등 사용 가능
) -> pd.DataFrame:
    """
    DB에서 indicator + forecast_date에 해당하는 데이터를 불러와
    (date, ticker) 중복을 집계한 후,
    date를 index, ticker를 column, value를 value로 하는 pivot DataFrame 생성
    """

    query = f"""
        SELECT date, ticker, value
        FROM {table_name}
        WHERE indicator = %s
          AND forecast_date = %s
        ORDER BY date, ticker
    """

    conn = pymysql.connect(
        host=db_info["host"],
        port=db_info.get("port", 3306),
        user=db_info["user"],
        password=db_info["password"],
        database=db_info["database"],
        charset="utf8mb4"
    )

    try:
        df = pd.read_sql(query, conn, params=[indicator, forecast_date])
    finally:
        conn.close()

    if df.empty:
        raise ValueError("조회된 데이터가 없습니다. indicator/forecast_date를 확인해주세요.")

    # (date, ticker) 중복 처리
    if agg in ("first", "last"):
        # 정렬 후 첫/마지막 값 사용
        df = df.sort_values(["date", "ticker"])
        grouped = df.groupby(["date", "ticker"], as_index=False)["value"].agg(agg)
    else:
        # 그 외 agg 함수(mean, sum 등)
        grouped = df.groupby(["date", "ticker"], as_index=False)["value"].agg(agg)

    # pivot
    pivot_df = grouped.pivot(index="date", columns="ticker", values="value")

    # index를 datetime으로 확실히 변환
    pivot_df.index = pd.to_datetime(pivot_df.index)

    return pivot_df


def calc_6month_growth_rate(pivot_df: pd.DataFrame) -> pd.DataFrame:
    df = pivot_df.copy()
    df.index = pd.to_datetime(df.index)

    # 숫자형 변환 (문자열로 들어왔을 경우 필수)
    df = df.apply(pd.to_numeric, errors='coerce')

    # ---- 6개월 전 대비 증감률 계산 ----
    # (현재값 - 6개월전값) / 6개월전값
    growth_6m = (df - df.shift(6)) / df.shift(6)

    # 마지막 시점 기준의 증감률만 추출 (가장 최근 row)
    latest_growth = growth_6m.iloc[-1].dropna()

    # 내림차순 정렬
    ranking_df = latest_growth.sort_values(ascending=False).reset_index()
    ranking_df.columns = ['ticker', 'growth_rate_6m']

    # Rank 부여
    ranking_df.insert(0, 'rank', range(1, len(ranking_df)+1))

    return ranking_df, growth_6m   # ranking + 전체 growth matrix도 반환



In [2]:
from DATA.stock_invest_function import get_db_host

db_info = {
    "host": get_db_host(),
    "port": 3307,
    "user": "stox7412",
    "password": "Apt106503!~",
    "database": "investar",
}

indicators = fetch_unique_indicators(
    db_info=db_info,
    table_name="Korea_company_valuation_ver2"
)

print(indicators)

['ensemble_forecast', 'ensemble_valuation', 'exog_var', 'exp_smoothing_forecast', 'forecast_date', 'is_forecast', 'lstm_forecast', 'lstm_valuation', 'matched_ttm_without_exog', 'matched_ttm_with_exog', 'mc_ets', 'mc_lstm', 'mc_prophet', 'mc_sarima_exog', 'mc_sarima_noexog', 'mc_theta', 'prophet_forecast', 'prophet_valuation', 'psr', 'psr_ETS', 'psr_LSTM', 'psr_Prophet', 'psr_SARIMA_exog', 'psr_SARIMA_noexog', 'psr_Theta', 'revenue_ensemble_forecast', 'revenue_ets', 'revenue_ets_ttm', 'revenue_exp_smoothing_forecast', 'revenue_lstm', 'revenue_lstm_forecast', 'revenue_lstm_ttm', 'revenue_prophet', 'revenue_prophet_forecast', 'revenue_prophet_ttm', 'revenue_sarima', 'revenue_sarima_exog', 'revenue_sarima_exog_ttm', 'revenue_sarima_ttm', 'revenue_theta', 'revenue_theta_ttm', 'revenue_without_exog_forecast', 'revenue_with_exog_forecast', 'sarima_forecast', 'sarima_valuation', 'ttm_revenue', 'ttm_revenue_without_exog', 'ttm_revenue_with_exog', 'valuation']


In [15]:
pivot_df = fetch_indicator_pivot(
    db_info=db_info,
    indicator= "revenue_sarima",
    forecast_date= "2026-02-04",
    ticker="A278470"
)

print(pivot_df)

indicator       revenue_sarima
date                          
2015-12-31                   0
2016-12-31                   0
2017-12-31                   0
2018-03-31         20436937000
2018-06-30         23044748000
2018-09-30         22344285000
2018-12-31         16430577000
2019-03-31         22621823000
2019-06-30         27196219660
2019-09-30         31675124540
2019-12-31         37233949110
2020-03-31         49441025000
2020-06-30         52117334000
2020-09-30         59659321000
2020-12-31         58721657000
2021-03-31         61914486000
2021-06-30         56332764000
2021-09-30         60557916000
2021-12-31         80340978000
2022-03-31         76342730000
2022-06-30         97942516000
2022-09-30         95300670000
2022-12-31        128112203000
2023-03-31        122178928000
2023-06-30        127672288000
2023-09-30        121938746000
2023-12-31        152019402550
2024-03-31        148928031350
2024-06-30        155493932210
2024-09-30        174117291160
2024-12-

In [9]:
psr_df = fetch_indicator_pivot(
    db_info=db_info,
    indicator= "mc_sarima_noexog",
    forecast_date= "2026-02-04",
    ticker="A278470"
)

print(psr_df)

indicator     mc_sarima_noexog
date                          
2026-03-31   8442722308813.129
2026-04-30   9160567740054.188
2026-05-31   11820794751920.24
2026-06-30   20111837662202.62
2026-07-31  23959436510606.836
2026-08-31   23789587.88582793
2026-09-30   33420442879950.12
2026-10-31   34088876023429.52
2026-11-30  29214621.467309397
2026-12-31   34036789.53789775
2027-01-31  14195552846929.713
2027-02-28  14717407457480.344
2027-03-31   22703960316421.23
2027-04-30  24634372520931.176
2027-05-31   31788189299558.52
2027-06-30   56216149431420.59
2027-07-31   66970869882478.79
2027-08-31   66496112.88455231
2027-09-30   96358456961439.66
2027-10-31      98285696122181
2027-11-30   84232152.62031515
2027-12-31  100643318.11403464


#### 매출증가율 순위 랭킹 부여

In [34]:
revenue_df = load_indicator_pivot(
    db_info=db_info,
    indicator="revenue_sarima",
    forecast_date="2026-01-26",
    agg="last"   # 필요하면 "mean" 등으로 변경 가능
)

In [22]:
revenue_df.tail(24)

ticker,A004000,A059210,A189300,A214150
date,,,,
2022-06-30,686259158820,17355693690,56984239970,32684579300
2022-09-30,628543197220,19142050610,57036789240,33268981200
2022-12-31,496608949000,17765704990,82419018170,40467758150
2023-03-31,524977420840,21164921400,64393461580,38974115960
2023-06-30,431178007410,19674701620,87594794970,45897817210
2023-09-30,401200689130,20940154710,66774944040,48246060900
2023-12-31,411264998330,21320944080,86279646460,47004803940
2024-03-31,399444149820,22502545390,46697934210,50379744700
2024-06-30,422072382130,25327086210,71699403380,58742159580


#### 매출액 증가 순위 추출

In [23]:
# ---------------- 사용 예시 ---------------- #
ranking_df, full_growth_matrix = calc_6month_growth_rate(revenue_df)
ranking_df.head(20)

,rank,ticker,growth_rate_6m
0,1,A004000,0.072221


In [26]:
pivot_df = load_indicator_pivot(
    db_info=db_info,
    indicator="mc_sarima_noexog",
    forecast_date="2026-01-26",
    agg="last"   # 필요하면 "mean" 등으로 변경 가능
)


In [33]:
pivot_df

ticker,A004000,A059210,A189300,A214150
date,,,,
2026-02-28,282008144595.12445,95894999642.24136,75230160175.25392,700361199152.149
2026-03-31,220944061120.58328,12613859818.137547,-271011303871.7828,576088388934.1368
2026-04-30,250915742359.73352,34710071647.10156,-130128373310.41928,1170996645441.7761
2026-05-31,259368212789.2302,95879012295.77794,267769486985.2705,987562052636.4482
2026-06-30,256573881172.19144,80586656537.11089,-38635318539.89769,1357642381806.7664
2026-07-31,256413864086.5627,44138588059.86168,-297257671083.6712,1260336911677.9475
2026-08-31,258375382884.94702,37377747233.92459,-62459764748.96248,-490477933771.7462
2026-09-30,261110454514.67087,82716454551.41821,248260287572.8355,813507218457.9512
2026-10-31,260742986745.0115,73813118816.71846,-150681762730.11987,707814081304.0199


#### 시장총액 증가율 예상

In [27]:
# ---------------- 사용 예시 ---------------- #
ranking_df, full_growth_matrix = calc_6month_growth_rate(pivot_df)

print(ranking_df.head(20))

   rank   ticker  growth_rate_6m
0     1  A004000        0.021225


In [30]:
def make_quarterly_df(pivot_df: pd.DataFrame) -> pd.DataFrame:
    df = pivot_df.copy()

    # 1) 인덱스를 날짜형으로 변환
    df.index = pd.to_datetime(df.index)

    # 2) 분기별 합산 (원래가 이미 분기 데이터라면, 분기당 1개 값이라 합과 동일)
    quarterly_df = df.resample('Q').sum(min_count=1)

    return quarterly_df

quarterly_df = make_quarterly_df(pivot_df)
# quarterly_data = quarterly_df.loc['2025' : '2026'].dropna(axis=1)

In [32]:
pivot_df

ticker,A004000,A059210,A189300,A214150
date,,,,
2026-02-28,282008144595.12445,95894999642.24136,75230160175.25392,700361199152.149
2026-03-31,220944061120.58328,12613859818.137547,-271011303871.7828,576088388934.1368
2026-04-30,250915742359.73352,34710071647.10156,-130128373310.41928,1170996645441.7761
2026-05-31,259368212789.2302,95879012295.77794,267769486985.2705,987562052636.4482
2026-06-30,256573881172.19144,80586656537.11089,-38635318539.89769,1357642381806.7664
2026-07-31,256413864086.5627,44138588059.86168,-297257671083.6712,1260336911677.9475
2026-08-31,258375382884.94702,37377747233.92459,-62459764748.96248,-490477933771.7462
2026-09-30,261110454514.67087,82716454551.41821,248260287572.8355,813507218457.9512
2026-10-31,260742986745.0115,73813118816.71846,-150681762730.11987,707814081304.0199


In [12]:
import pandas as pd
import numpy as np

def rank_growth_2026_vs_2025(quarterly_df: pd.DataFrame) -> pd.DataFrame:
    df_q = quarterly_df.copy()
    df_q.index = pd.to_datetime(df_q.index)

    # 🔹 0) 문자열 → 숫자형으로 변환 (핵심!)
    df_q = df_q.apply(pd.to_numeric, errors='coerce')

    # 1) 연간 매출 = 분기 매출 합산
    annual_df = df_q.resample('Y').sum(min_count=1)
    annual_df.index = annual_df.index.year  # 2004-12-31 → 2004

    # 2) 2025년, 2026년 데이터 체크
    if 2025 not in annual_df.index or 2026 not in annual_df.index:
        raise ValueError("annual_df에 2025 또는 2026 데이터가 없습니다.")

    rev_2025 = annual_df.loc[2025]
    rev_2026 = annual_df.loc[2026]

    # 🔹 2-1) 2025 또는 2026이 전부 NaN인 컬럼 제거 (선택적이지만 안전)
    both_years = pd.concat([rev_2025, rev_2026], axis=1)
    both_years.columns = ['rev_2025', 'rev_2026']
    both_years = both_years.dropna(how='all')

    rev_2025 = both_years['rev_2025']
    rev_2026 = both_years['rev_2026']

    # 3) 성장률 = (2026 - 2025) / 2025
    #    2025 매출이 0이거나 NaN이면 제거
    valid_mask = (rev_2025.notna()) & (rev_2026.notna()) & (rev_2025 != 0)
    rev_2025 = rev_2025[valid_mask]
    rev_2026 = rev_2026[valid_mask]

    growth_2026 = (rev_2026 - rev_2025) / rev_2025

    # 4) 내림차순 정렬 + 랭킹
    ranking_df = growth_2026.sort_values(ascending=False).reset_index()
    ranking_df.columns = ['ticker', 'growth_2026_vs_2025']

    ranking_df.insert(0, 'rank', range(1, len(ranking_df) + 1))

    return ranking_df


# 사용 예시
quarterly_df = make_quarterly_df(pivot_df)   # 1번 함수 재사용

quarterly_data = quarterly_df.loc['2025' : '2026'].dropna(axis=1)

ranking_df = rank_growth_2026_vs_2025(quarterly_data)


ValueError: annual_df에 2025 또는 2026 데이터가 없습니다.

In [64]:
ranking_df

,rank,ticker,growth_2026_vs_2025
0,1,A066970,1.029482
1,2,A059090,0.410540
2,3,A031980,0.363870
3,4,A012450,0.328059
4,5,A214150,0.327559
...,...,...,...
59,60,A123700,0.010343
60,61,A002350,0.008477
61,62,A073240,0.004079
62,63,A043150,-0.001202


In [56]:
from datetime import datetime
import os

# 저장 경로
path = r"C:\Users\82108\OneDrive\바탕 화면\investment\data\analysis_results"

# 데이터 (예시)
q_data = quarterly_data.T
# 오늘 날짜 생성
today = datetime.today().strftime("%Y%m%d")
# 파일명 생성
file_name = f"revenue_growth_sarima_{today}.xlsx"
# 전체 파일 경로
full_path = os.path.join(path, file_name)
# 저장
q_data.to_excel(full_path, index=True)
print(f"파일 저장 완료: {full_path}")

파일 저장 완료: C:\Users\82108\OneDrive\바탕 화면\investment\data\analysis_results\revenue_growth_sarima_20251226.xlsx


In [52]:
ranking_df

,rank,ticker,growth_2026_vs_2025
0,1,A066970,0.579162
1,2,A025980,0.525452
2,3,A059090,0.522349
3,4,A031980,0.422202
4,5,A032500,0.412110
...,...,...,...
62,63,A123700,0.002631
63,64,A006910,-0.029605
64,65,A011780,-0.039003
65,66,A051910,-0.067038


In [3]:
# import pymysql
#
# conn = pymysql.connect(**db_info)
# cursor = conn.cursor()
#
# try:
#     # 삭제 전 데이터 개수 확인
#     check_sql = """
#     SELECT COUNT(*) as count
#     FROM Korea_company_valuation_ver2
#     WHERE forecast_date = '2025-12-25'
#     """
#     cursor.execute(check_sql)
#     count_before = cursor.fetchone()[0]
#     print(f"삭제 전 데이터 개수: {count_before:,}")
#
#     # 데이터 삭제
#     delete_sql = """
#     DELETE FROM Korea_company_valuation_ver2
#     WHERE forecast_date = '2025-12-26'
#     """
#     cursor.execute(delete_sql)
#     conn.commit()
#
#     deleted_rows = cursor.rowcount
#     print(f"삭제된 데이터 개수: {deleted_rows:,}")
#
#     # 삭제 후 확인
#     cursor.execute(check_sql)
#     count_after = cursor.fetchone()[0]
#     print(f"삭제 후 남은 데이터 개수: {count_after:,}")
#
# except Exception as e:
#     conn.rollback()
#     print(f"오류 발생: {e}")
#
# finally:
#     cursor.close()
#     conn.close()
#
# print("\n삭제 완료")



삭제 전 데이터 개수: 7,390
삭제된 데이터 개수: 49,215
삭제 후 남은 데이터 개수: 7,390

삭제 완료
